In [1]:
from prompt_utils import build_prompt
from data_utils import create_submission, parse_predictions, create_comparison_df, read_dataset
from semevalpolar.llm import custom_rules
from semevalpolar.llm.main import test_run, create_gen, create_response
from tqdm import tqdm
from semevalpolar.utils import get_project_root

from custom_rules import apply_rules

import os
import ast

D:\projects\mldl\semevalpolar


In [2]:
batch_size = 10
data_path = os.path.join(get_project_root(), 'data', 'dev_phase', 'subtask1', 'train', 'eng.csv')
gen = create_gen(data_path, batch_size=batch_size, randomize=True)
generator_list = list(gen)

In [3]:
df = read_dataset(data_path)

In [4]:
predictions = []
usages = []

In [5]:
for batch in tqdm(generator_list[:30]):
    response = test_run(batch, prompt_path="prompt-ds-main-classifier.txt")
    predictions.append(parse_predictions(response.output_text))
    usages.append(response.usage)

100%|██████████| 30/30 [05:20<00:00, 10.70s/it]


In [6]:
len(predictions)

30

In [7]:
flat = [x for sub in predictions for x in sub]

In [8]:
ground_truths = []

In [9]:
for i in range(len(flat) // len(predictions)):
    for x in generator_list[i]["polarization"]:
        ground_truths.append(x)

In [21]:
texts = []

for i in range(len(predictions)):
    for j in range(batch_size):
        texts.append(generator_list[i]["text"].iloc[j])

In [23]:
from sklearn.metrics import confusion_matrix

comparison = create_comparison_df(flat, ground_truths, texts)
cm = confusion_matrix(comparison["Ground Truth"], comparison["Predicted"], labels=[0, 1])
cm

array([[57,  5],
       [25, 13]])

In [11]:
from sklearn.metrics import accuracy_score

accuracy_score(comparison["Ground Truth"], comparison["Predicted"])

0.7

In [24]:
comparison

,Predicted,Ground Truth,Text
0,0,1,Imagine blaming this on Israel instead of Hamas
1,1,1,Blinken is the Butcher of Blinken ButcherOfGaz...
2,0,0,"And as far as your gun control, Chicago knows ..."
3,0,0,All the spoilers are on Fox News daily.
4,0,1,Are red states stealing from California?
...,...,...,...
95,0,0,Israel at war after Hamas launches surprise at...
96,0,1,Well see how this story ends. Theres a conside...
97,0,0,Texas Republicans Target Climate Science in Te...
98,0,0,US seeks Israeli approval to arm PA troops in ...


In [25]:
wrong = comparison[comparison["Ground Truth"] != comparison["Predicted"]]
wrong

,Predicted,Ground Truth,Text
0,0,1,Imagine blaming this on Israel instead of Hamas
4,0,1,Are red states stealing from California?
7,0,1,"""Sufficient evidence of voter fraud?"" Where? m..."
9,0,1,"Yeah, some IDF are indistinguishable from them."
12,0,1,House Republicans plot impeachment against jud...
13,0,1,I just signed a petition Investigate X and Elo...
14,0,1,And they will call it election interference an...
16,0,1,Lying. Lots of election interference and you k...
20,0,1,Wheres the results?!? Rigged Election! ucpldr
22,0,1,"Has Israel considered not being a genocidal, a..."


In [29]:
submission = create_submission(df, flat)

In [30]:
submission.to_csv(os.path.join(get_project_root(), 'predictions', 'subtask_1', 'pred_eng.csv'), index=False)